# sum-and-broadcast-duality — worked example 1: sum_back: re-insert axis then expand

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sum-and-broadcast-duality`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

`sum` is a linear map whose transpose is broadcast, so the backward of `out = x.sum(dim)` is: re-insert the dropped axis (if keepdim was False) with `unsqueeze`, then broadcast back to `x.shape`. The `.clone()` materializes a contiguous tensor since `expand_as` returns a zero-stride view.

## Worked solution

The forward sums a `(4, 3)` tensor along `dim=1` (keepdim False), producing `(4,)`. In `sum_back` we `unsqueeze(1)` to get `(4, 1)`, then `expand_as(x).clone()` to spread the gradient uniformly across the summed axis back to `(4, 3)`. Each input position gets the same upstream gradient because summing distributes its gradient equally. We verify against autograd on the same forward and print the shape and a sample row.

In [ ]:
Tensor = t.Tensor


def sum_back(grad_out, out, x, dim, keepdim=False):
    if not keepdim:
        grad_out = grad_out.unsqueeze(dim)
    return grad_out.expand_as(x).clone()


t.manual_seed(0)
x = t.randn(4, 3)
out = x.sum(dim=1)
grad_out = t.tensor([1.0, 2.0, 3.0, 4.0])
grad_x = sum_back(grad_out, out, x, dim=1)
print('grad_x shape:', tuple(grad_x.shape))

xg = x.clone().requires_grad_(True)
xg.sum(dim=1).backward(grad_out)
print('matches autograd:', bool(t.allclose(grad_x, xg.grad)))